# exploring cloud reading of data with icepyx

in preparing for the April 2026 cryo hack, I played around with several methods of reading data in the cloud (xarray + h5coro, h5coro directly, an AI generated, generalized version of Tyler's ATL03 granule read-in function...)

this notebook enables running those (along with the temporary `readdev.py` script), loosely providing a way to run analogous versions locally and in the cloud in case we want to compare them.

Removed from history/setting block of presentation, but would like to add to table someday (along with expanding to include other efforts like SR, Veda, etc.)

### Explorations and Experiments
- Rachel's discussions (fsspec, etc.)
- Aimee's geoparquet implementation
- Luis, Andy, Amy whitepaper
- Hackweek collaborations (2022 on)

In [ ]:
%pip install -e ~/icepyx

In [ ]:
import icepyx as ipx
ipx.__version__

In [ ]:
%load_ext autoreload
import icepyx as ipx
%autoreload 2

from icepyx.core.variables import Variables as Variables
from icepyx.core.variables import list_of_dict_vals

import icepyx.core.readdev as readdev

In [ ]:
import datashader
import geoviews as gv
import hvplot.xarray
import matplotlib.pyplot as plt

In [ ]:
%matplotlib inline

In [ ]:
# Use our search parameters to setup a search Query
short_name = 'ATL06'
spatial_extent = [-39, 66.2, -37.7, 66.6]
date_range = ['2019-05-04','2019-07-04'] #'2019-08-04' was orig example value used # '2019-05-10' will get you one granule only
region = ipx.Query(short_name, spatial_extent, date_range)

In [ ]:
# Visualize our spatial extent
region.visualize_spatial_extent()

In [ ]:
# Display if any data files, or granules, matched our search
region.avail_granules(ids=True)

In [ ]:
# We can also get the S3 urls
region.avail_granules(ids=True, cloud=True)

In [ ]:
s3urls = region.avail_granules(ids=True, cloud=True)[1]

In [ ]:
path_root = "/Users/jessica/computing/icepyx/test_data/"

In [ ]:
# download a file and use it for local reads

region.download_granules(path=path_root)

## Reading a file with icepyx

To read a file with icepyx there are several steps:
1. Create a `Read` object. This sets up an initial connection to your file(s) and validates the metadata.
2. Tell the `Read` object what variables you would like to read
3. Load your data!

### Create a `Read` object

Here we are creating a read object to set up an initial connection to your file(s).
It will ask you if you'd like to proceed - enter "y".

In [ ]:
# local runs!
reader = ipx.Read(path_root+"*ATL06*007*")

In [ ]:
# cloud runs!
reader = ipx.Read(s3urls)

<div class="alert alert-block alert-info">
<b>Tip:</b> If you don't want to type your Earthdata Login information every time they are
    required you can setup more automatic methods of authentication. Two common methods
    are 1) Add your earthdata password and username to as environment variables
    as EARTHDATA_USERNAME and EARTHDATA_PASSWORD. 2) setup a .netrc file in your home directory. See <a href="https://nasa-openscapes.github.io/2021-Cloud-Hackathon/tutorials/04_NASA_Earthdata_Authentication.html"> the Openscapes tutorial</a> </div>

In [ ]:
reader.filelist

### Select your variables

To view the variables contained in your dataset you can call `.vars` on your data reader.

In [ ]:
reader.variables.avail()

In [ ]:
# fewer beam version
reader.variables.append(beam_list=['gt2l','gt3l','gt2r'], var_list=['h_li', 'latitude', 'longitude', 'delta_time'])
# reader.variables.append(beam_list=['gt3l','gt2l'], keyword_list=['land_ice_segments'], var_list=['delta_ti

In [ ]:
# all beams (default)
reader.variables.append(var_list=['h_li', 'latitude', 'longitude'])
reader.variables.append(keyword_list=['land_ice_segments'], var_list=['delta_time'])

Note that adding variables is a required step before you can load the data.

In [ ]:
reader.variables.wanted

In [ ]:
reader.variables.remove(all=True)

### Load the data!

In [ ]:
%%time
ds = reader.load()

# TODO: use s3pathlib or something more robust to strip the s3 prefix for h5coro version

# output from local run
# CPU times: user 2.8 s, sys: 313 ms, total: 3.11 s
# Wall time: 3.27 s

# BUG: cannot figure out where the "default" variables are added
# (they appear in reader.variables.wanted only after reader.load() is called)
# but for some reason 'delta_time' isn't being added
# manually adding delta_time still results in an error, so there's something else going on
# key transitions from non-error to error seem to be number (or which) granules (6 works, 10 doesn't)
# and number of beams (two work, all don't)

In [ ]:
ds

In [ ]:
s3url_atl06 = 'nsidc-cumulus-prod-protected/ATLAS/ATL06/007/2019/05/09/ATL06_20190509202526_06350305_007_01.h5'

In [ ]:
import earthaccess

auth = earthaccess.login()
creds = auth.get_s3_credentials(daac='NSIDC')


In [ ]:
%%time
# xarray backend can only read groups with variables with uniform lengths.
# Thus, `group='orbit_info'` will fail.
# Adding the `pick_variables=[]` kwarg, e.g. with 'sc_orient_time', "works" but never actually provides any data
# however, passing in "sc_orient" works, and loads a dimension without coordinates of "sc_orient_time"
# This matches the use of DIMENSIONS in how the data is stored, as viewed in the hdf5 viewer

import xarray as xr
ds = xr.open_dataset(s3url_atl06, 
                     engine='h5coro', 
                     group='ancillary_data',
                     pick_variables=["data_start_utc"],
                     credentials=creds)
ds

In [ ]:
ds["data_start_utc"]

In [ ]:
%%time
# xarray backend can only read groups with variables with uniform lengths.
# Thus, `group='orbit_info'` will fail.
# Adding the `pick_variables=[]` kwarg, e.g. with 'sc_orient_time', "works" but never actually provides any data

import xarray as xr
ds = xr.open_dataset(s3url_atl06, engine='h5coro', group='gt2l/land_ice_segments', credentials=creds)
ds

In [ ]:
ds.sc_orient.values

In [ ]:
from h5coro import h5coro, s3driver

# (2) create
h5obj = h5coro.H5Coro(f'{s3url_atl06}', s3driver.S3Driver, credentials=creds)

# (3) read
datasets = [{'dataset': 'orbit_info/sc_orient', 'hyperslice': []},
             {'dataset': '/gt2l/land_ice_segments/latitude', 'hyperslice': []}]
promise = h5obj.readDatasets(datasets=datasets, block=True)
# hyperslice allows subsetting of data (think array indices, needed for multidimensional data; JP says can probably skip providing

# (4) display
for variable in promise:
    print(f'{variable}: {promise[variable]}')

In [ ]:
promise['orbit_info/sc_orient']

In [ ]:
ds

In [ ]:
# read it in using the new reader function
IS2_atl03_mdsAI, IS2_atl03_attrsAI, IS2_atl03_beamsAI = readdev.read_granule(reader.filelist[0], 
                                                                                 ATTRIBUTES=True,
                                                                                 variables_obj=reader.variables
                                                                                 )


In [ ]:
IS2_atl03_mdsAI

In [ ]:
IS2_atl03_attrsAI

Here we have an xarray Dataset, a common Python data structure for analysis. To visualize the data we can plot it using:

In [ ]:
# single mabplotlib axis
ax = ds.isel(gran_idx=0).plot.scatter(x="longitude", y="latitude", hue="h_li")

In [ ]:
# recreate the above plot using the dicts instead
import numpy as np
lons = IS2_atl03_mdsAI["gt2l"]['land_ice_segments']["longitude"]
lons = np.append(lons,IS2_atl03_mdsAI["gt3l"]['land_ice_segments']["longitude"])
lats = IS2_atl03_mdsAI["gt2l"]['land_ice_segments']["latitude"]
lats = np.append(lats, IS2_atl03_mdsAI["gt3l"]['land_ice_segments']["latitude"])
hts = IS2_atl03_mdsAI["gt2l"]['land_ice_segments']["h_li"]
hts = np.append(hts, IS2_atl03_mdsAI["gt3l"]['land_ice_segments']["h_li"])

In [ ]:
ax2 = plt.scatter(x=lons, y=lats, c=hts)

# going to take some work to get the legend there

NEXT STEPS:
- compare local and cloud versions
(in progress) - try with > 1 granules (and/or more variables)
- add a legend/z values to the plot
- see about polar projection plotting
- figure out why there are so many more points in the dict version (no data value?)
- get a legend to plot on the dict version
- X try the h5coro engine version in the cloud
- X modify read function and try an h5coro read version in the cloud

OPTIONS (for what to do next):
- time local and cloud versions (for xarray and dict option)
- add more beams
- try a different granule
- try putting together multiple granules

It would be great if we could see the data on a background map.
To do this, we'll use the GeoViews library.

<div class="alert alert-block alert-info">
<b>Notice:</b> The interactive plots are not rendered within this Jupyterbook because the are too big to include in a JupyterNotebook on GitHub.</a> </div>

In [ ]:
import cartopy.crs as ccrs

In [ ]:
ccrs.epsg(4326)

In [ ]:
tile = gv.tile_sources.EsriImagery.opts(width=500, height=500)
tile

In [ ]:
tile2 = gv.tile_sources.EsriArcticImagery(projection=ccrs.NorthPolarStereo()).opts(width=500, height=500)
tile2

First, we must convert our data to geodetic coordinates and add them to the DataSet.

In [ ]:
print(spatial_extent)

In [ ]:
# subset ds to spatial extent (since cloud automatically streams whole granules)
ds_sub = ds.sel(lat=slice(spatial_extent[1],spatial_extent[3]), lon=slice(spatial_extent[0],spatial_extent[2])) 

In [ ]:
x, y = datashader.utils.lnglat_to_meters(ds.longitude, ds.latitude)

In [ ]:
ds = ds.assign(x=x, y=y)

In [ ]:
# compute x and y min and max for plotting
xmin, ymin = datashader.utils.lnglat_to_meters(spatial_extent[0], spatial_extent[1])
xmax, ymax = datashader.utils.lnglat_to_meters(spatial_extent[2], spatial_extent[3])

In [ ]:
# create our plot (locally)
is2 = ds.hvplot.scatter(x="x", y="y", groupby=["data_start_utc"], rasterize=True)

In [ ]:
# create our plot (cloud)
is2 = ds.hvplot.scatter(x="x", 
                        y="y", 
                        groupby=[], #"rgt"], 
                        # rasterize=True,
                        xlim=(xmin, xmax),
                        ylim=(ymin, ymax),
                        color="h_li"
                       )

In [ ]:
is2

In [ ]:
# background via geoviews
is2 * tile